# ???????????? 

In [ ]:
import pandas as pd
import datetime
import numpy as np
from sklearn import preprocessing
from scipy.sparse import csr_matrix

## ???????? ???????? ??????

??? ???????? ???????????? ?????? ??????? events:

In [ ]:
events = pd.read_csv('events.csv')
print(events.shape)
display(events)

## ????????????? ????????? ????????: 

?????????? ? ??????? ? ?????? datatime:

In [ ]:
events = events.assign(date=pd.Series(datetime.datetime.fromtimestamp(i/1000).date() for i in events.timestamp))
events = events.sort_values('date').reset_index(drop=True)
events = events[['visitorid','itemid','event', 'date']]
events.head(5)

????????? ?????? ?? ??????? ???????:

In [ ]:
start_date = '2015-5-3'
end_date = '2015-5-18'
fd = lambda x: datetime.datetime.strptime(x, '%Y-%m-%d').date()
events = events[(events.date >= fd(start_date)) & (events.date <= fd(end_date))]

??????? ??????? ?? ???????? ? ???? ??? ?????????:

In [ ]:
split_point = int(np.round(events.shape[0]*0.8))
events_train = events.iloc[0:split_point]
events_test = events.iloc[split_point::]
events_test = events_test[(events_test['visitorid'].isin(events_train['visitorid']))
                          & (events_test['itemid'].isin(events_train['itemid']))]

???????? ????????, ????? ???????????? ?? ? ???????? ??????:

In [ ]:
id_cols=['visitorid','itemid']
trans_cat_train=dict()
trans_cat_test=dict()

for k in id_cols:
    cate_enc=preprocessing.LabelEncoder()
    trans_cat_train[k]=cate_enc.fit_transform(events_train[k].values)
    trans_cat_test[k]=cate_enc.transform(events_test[k].values)
    
ratings = dict()

cate_enc=preprocessing.LabelEncoder()
ratings['train'] = cate_enc.fit_transform(events_train.event)
ratings['test'] = cate_enc.transform(events_test.event)    

??????? item-users sparse-???????:

In [ ]:
n_users=len(np.unique(trans_cat_train['visitorid']))
n_items=len(np.unique(trans_cat_train['itemid']))

rate_matrix = dict()

#?????????? ??????????? ???????
rate_matrix['train'] = csr_matrix((ratings['train'],
                                   (trans_cat_train['visitorid'],
                                    trans_cat_train['itemid'])),
                                  shape=(n_users,n_items)
                                  )

rate_matrix['test'] = csr_matrix((ratings['test'],
                                  (trans_cat_test['visitorid'],
                                   trans_cat_test['itemid'])),
                                 shape=(n_users,n_items)
                                 )

## ????????? ?????? ? ??????? ????????

? ???????? ?????? ?????????????? ALS:

In [ ]:
# ??????
from implicit.als import AlternatingLeastSquares
# ??? ?????? ??????
from implicit.evaluation import mean_average_precision_at_k
from implicit.evaluation import AUC_at_k

In [ ]:
ALS_model = AlternatingLeastSquares(factors=10, random_state=42)
ALS_model.fit(rate_matrix['train'])

????????? ????????????:

In [ ]:
user_id = 1 # id ????????????
unique_items = np.unique(trans_cat_train['itemid']) # ?????? ???????

recomendations_ids, scores = ALS_model.recommend(user_id, rate_matrix['train'][user_id])
recomendations = unique_items[recomendations_ids]
print('Recomendations for user {}: {}'.format(user_id, recomendations))

???????:

In [ ]:
map_at3 = mean_average_precision_at_k(ALS_model, rate_matrix['train'], rate_matrix['test'], K=3)
print('Mean Average Precision at 3: {:.5f}'.format(map_at3))
print('AUC_at_3:', AUC_at_k(ALS_model, rate_matrix['train'], rate_matrix['test'], K=3))

## ???????? docker-?????? ? ???????

??? ??? ????????????? ?????????? ??????? ? ???????:

In [ ]:
#from flask import Flask, request, jsonify
#import pickle
#import numpy as np

#with open(r'ALS_model.pkl', 'rb') as pkl_file1: 
#    model = pickle.load(pkl_file1)

#with open(r'unique_items.pkl', 'rb') as pkl_file2: 
#    unique_items = pickle.load(pkl_file2)
    
#with open(r'rate_matrix.pkl', 'rb') as pkl_file3: 
#    rate_matrix = pickle.load(pkl_file3)    
    
#app = Flask(__name__)

#@app.route('/recommendation')
#def hello_func():
#    user = int(request.args.get('name'))
#    recomendations_ids, scores = model.recommend(user, rate_matrix[user])
#    recomendations = unique_items[recomendations_ids]
#    return f'recommendation for user {user}:{recomendations[:3]}!'


#if __name__ == '__main__':
#    app.run(host='0.0.0.0', port=5000)

Docker-????? ?????????? ?? ???????????? Dockerfile(docker build -t diplom .). ? ????? requirements ??????? ??????????? ??????????. ? ????? app ????? ???? ???????, ??????????? ?????? ? ??????????? ??????? ? ??????? pkl.
??????????? ?????? ? ??????????:
 
  
diplom

  ??app

     ??ALS_model.pkl

     ??rate_matrix.pkl

     ??unique_items.pkl

     ??server.py

  ??Dockerfile
  
  ??requirements.txt
  
????????? ??????????? ?? ????????? ??????: docker run -it --rm --name=diplom -p=5000:5000 diplom
????? ?? ?????? http://127.0.0.1:5000/recommendation?user_id=35 ????? ???????? ???????????? ???? ??????? ??? ??? ?????????? user_id(??????????? ??? /recommendation?user_id=?????_?????)

???? server.py ????? ????????? ???????? ?? docker-?????? ? ????? ???????? ???????????? ?? ???????.

?????? ??????:

![picture_of_bird](1.PNG)

?????? ????????? ????????????:

![picture_of_bird](2.PNG)
![picture_of_bird](3.PNG)